# 15 — Pseudo-Labeling and Semi-Supervised Learning (Lesson)

## Problem Definition
Leverage unlabeled data without reinforcing model errors.

## Required Prior Knowledge
- Notebooks 12 and 14 for reliable OOF validation and ensembling strategy.
- Classification losses and confidence interpretation.

## New Concepts Introduced
- Semi-supervised objective decomposition.
- Confidence-threshold pseudo-label rule.
- Confirmation-bias risk derivation.
- Iterative pseudo-label loop with diagnostics.

### Mathematical Foundations
Semi-supervised objective:
$$
\\mathcal{L}(\\theta)=\\mathcal{L}_{sup}(\\theta)+\\lambda\\,\\mathcal{L}_{unsup}(\\theta).
$$
For pseudo-labeling with threshold $\\tau$:
$$
\\tilde{y}_u = \\mathbf{1}[p_\\theta(y=1\\mid x_u)\\ge \\tau],\\qquad
\\mathcal{U}_\\tau=\\{u:\\max_c p_\\theta(c\\mid x_u)\\ge\\tau\\}.
$$

Unsupervised term:
$$
\\mathcal{L}_{unsup}(\\theta)=\\frac{1}{|\\mathcal{U}_\\tau|}\\sum_{u\\in\\mathcal{U}_\\tau}
\\ell\\big(f_\\theta(x_u),\\tilde{y}_u\\big).
$$

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

### Step-by-Step Derivation
1. Start from supervised empirical risk on labeled set $\\mathcal{D}_L$:
   $$
   \\mathcal{L}_{sup}=\\frac{1}{|\\mathcal{D}_L|}\\sum_{(x,y)\\in\\mathcal{D}_L}\\ell(f_\\theta(x),y).
   $$
2. Generate pseudo-labels on unlabeled pool $\\mathcal{D}_U$ using current model.
3. Keep only confident points $\\mathcal{U}_\\tau$.
4. Add pseudo-label loss with multiplier $\\lambda$.
5. Confirmation bias risk: if pseudo-label error rate is $\\epsilon$, expected noisy contribution scales as
   $$
   \\lambda\\,\\epsilon\\,\\mathbb{E}[\\Delta\\ell],
   $$
   so large $\\lambda$ and low $\\tau$ can amplify wrong gradients.

## Intuition
Pseudo-labeling is self-training; confidence filtering trades data quantity for label quality.

## Mapping from Math to Implementation
- We maintain separate labeled/unlabeled splits.
- At each iteration, high-confidence unlabeled samples are added with pseudo-labels.
- Metrics track gain versus confirmation-bias drift.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=10, class_sep=1.0, flip_y=0.03, random_state=SEED
)
X_lab, X_pool, y_lab, y_pool_true = train_test_split(X, y, test_size=0.7, random_state=SEED, stratify=y)
X_unl, X_test, y_unl_true, y_test = train_test_split(X_pool, y_pool_true, test_size=0.4, random_state=SEED, stratify=y_pool_true)

clf = LogisticRegression(max_iter=2000).fit(X_lab, y_lab)
base_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

tau = 0.9
for step in range(3):
    proba = clf.predict_proba(X_unl)[:, 1]
    conf = np.maximum(proba, 1 - proba)
    mask = conf >= tau
    pseudo = (proba[mask] >= 0.5).astype(int)
    if mask.sum() == 0:
        break
    X_lab = np.vstack([X_lab, X_unl[mask]])
    y_lab = np.concatenate([y_lab, pseudo])
    X_unl = X_unl[~mask]
    clf = LogisticRegression(max_iter=2000).fit(X_lab, y_lab)

pl_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print({"baseline_auc": base_auc, "pseudo_label_auc": pl_auc, "remaining_unlabeled": len(X_unl)})

## Real Dataset Experiment

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data.values
y = (wine.target.values == 2).astype(int)
X_lab, X_unl, y_lab, y_unl_true = train_test_split(X, y, test_size=0.6, random_state=SEED, stratify=y)
X_unl_pool, X_test, y_unl_pool_true, y_test = train_test_split(X_unl, y_unl_true, test_size=0.4, random_state=SEED, stratify=y_unl_true)

base = HistGradientBoostingClassifier(random_state=SEED).fit(X_lab, y_lab)
base_auc = roc_auc_score(y_test, base.predict_proba(X_test)[:, 1])

proba = base.predict_proba(X_unl_pool)[:, 1]
mask = np.maximum(proba, 1 - proba) >= 0.92
pseudo = (proba[mask] >= 0.5).astype(int)
X_aug = np.vstack([X_lab, X_unl_pool[mask]])
y_aug = np.concatenate([y_lab, pseudo])
aug = HistGradientBoostingClassifier(random_state=SEED).fit(X_aug, y_aug)
aug_auc = roc_auc_score(y_test, aug.predict_proba(X_test)[:, 1])
print({"wine_base_auc": base_auc, "wine_pseudo_auc": aug_auc, "pseudo_points": int(mask.sum())})

## Diagnostic Analysis

In [ ]:
conf = np.maximum(proba, 1 - proba)
plt.figure(figsize=(6, 3))
plt.hist(conf, bins=30)
plt.title("Unlabeled confidence distribution")
plt.xlabel("max class probability")
plt.tight_layout()
plt.show()

if mask.sum() > 0:
    pseudo_error = float(np.mean(pseudo != y_unl_pool_true[mask]))
    print({"pseudo_label_error_rate": pseudo_error})

## Failure Case Demonstration

In [ ]:
# Failure case: low threshold increases noisy pseudo-labels.
low_tau = 0.55
mask_low = np.maximum(proba, 1 - proba) >= low_tau
pseudo_low = (proba[mask_low] >= 0.5).astype(int)
X_low = np.vstack([X_lab, X_unl_pool[mask_low]])
y_low = np.concatenate([y_lab, pseudo_low])
model_low = HistGradientBoostingClassifier(random_state=SEED).fit(X_low, y_low)
auc_low = roc_auc_score(y_test, model_low.predict_proba(X_test)[:, 1])
print({"low_threshold_auc": auc_low, "high_threshold_auc": aug_auc, "low_tau_points": int(mask_low.sum())})

## Exercise Ladder (basic → advanced → research-level)
1. Derive gradient contribution of pseudo-label noise under logistic loss.
2. Compare fixed threshold vs quantile-based threshold schedules.
3. Add teacher-student ensembling for pseudo labels.
4. Analyze when pseudo-labeling hurts under class imbalance.

## Summary of Mathematical Insights
Pseudo-labeling is useful only with confidence discipline and strict monitoring of pseudo-label error propagation.